In [ ]:
# Install LibreYOLO for checkpoint serialization
try:
    import libreyolo
except ImportError:
    print("Cloning and installing libreyolo...")
    !git clone -b librefomo https://github.com/bencejdanko/libreyolo.git
    !pip install -e ./libreyolo


In [13]:
import importlib.util
import math
import sys
import urllib.request
from collections import OrderedDict
from pathlib import Path
from typing import List

import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn


# ----------------------------
# Config
# ----------------------------

OUTDIR = Path("/content/librefomo_export")
OUTDIR.mkdir(parents=True, exist_ok=True)

SEED = 123
VERIFY_ATOL_MAX = 1e-3

SIZES = ["s", "m", "l"]

CONFIGS = {
    "s": {
        "alpha": 0.35,
        "imgsz": 96,
        "head_in_channels": 96,
        "url": "https://storage.googleapis.com/tensorflow/keras-applications/mobilenet_v2/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_0.35_96_no_top.h5",
        "out": "LibreFOMOs.pt",
    },
    "m": {
        "alpha": 0.50,
        "imgsz": 192,
        "head_in_channels": 96,
        "url": "https://storage.googleapis.com/tensorflow/keras-applications/mobilenet_v2/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_0.5_192_no_top.h5",
        "out": "LibreFOMOm.pt",
    },
    "l": {
        "alpha": 1.00,
        "imgsz": 224,
        "head_in_channels": 192,
        "url": "https://storage.googleapis.com/tensorflow/keras-applications/mobilenet_v2/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5",
        "out": "LibreFOMOl.pt",
    },
}

np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)
torch.set_grad_enabled(False)


# ----------------------------
# Write clean runtime-only model.py
# ----------------------------

MODEL_PY = r'''
import math
from typing import Tuple

import torch
import torch.nn as nn


CONFIGS = {
    "s": {"alpha": 0.35, "imgsz": 96, "head_in_channels": 96},
    "m": {"alpha": 0.50, "imgsz": 192, "head_in_channels": 96},
    "l": {"alpha": 1.00, "imgsz": 224, "head_in_channels": 192},
}


def _make_divisible(v: float, divisor: int = 8, min_value=None):
    if min_value is None:
        min_value = divisor

    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)

    if new_v < 0.9 * v:
        new_v += divisor

    return new_v


def _same_pad_1d(input_size: int, kernel: int, stride: int, dilation: int = 1):
    out_size = math.ceil(float(input_size) / float(stride))
    effective_kernel = (kernel - 1) * dilation + 1
    total_pad = max((out_size - 1) * stride + effective_kernel - input_size, 0)

    before = total_pad // 2
    after = total_pad - before

    return before, after


def _same_pad_2d(input_hw: Tuple[int, int], kernel_hw, stride_hw, dilation_hw=(1, 1)):
    ih, iw = input_hw
    kh, kw = kernel_hw
    sh, sw = stride_hw
    dh, dw = dilation_hw

    top, bottom = _same_pad_1d(ih, kh, sh, dh)
    left, right = _same_pad_1d(iw, kw, sw, dw)

    return left, right, top, bottom


class StaticSamePadConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size,
        stride=1,
        groups: int = 1,
        bias: bool = False,
        input_hw: Tuple[int, int] = None,
    ):
        super().__init__()

        if input_hw is None:
            raise ValueError("StaticSamePadConv2d requires fixed input_hw")

        kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        stride = stride if isinstance(stride, tuple) else (stride, stride)

        self.input_hw = tuple(input_hw)
        self.kernel_size = tuple(kernel_size)
        self.stride = tuple(stride)
        self.dilation = (1, 1)

        self.pad = nn.ZeroPad2d(
            _same_pad_2d(
                input_hw=self.input_hw,
                kernel_hw=self.kernel_size,
                stride_hw=self.stride,
                dilation_hw=self.dilation,
            )
        )

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=0,
            dilation=1,
            groups=groups,
            bias=bias,
        )

    def forward(self, x):
        return self.conv(self.pad(x))


class ConvBNReLU6(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size, stride, input_hw):
        super().__init__(
            StaticSamePadConv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                groups=1,
                bias=False,
                input_hw=input_hw,
            ),
            nn.BatchNorm2d(out_channels, eps=1e-3),
            nn.ReLU6(inplace=True),
        )


class DepthwiseConvBNReLU6(nn.Sequential):
    def __init__(self, channels, kernel_size, stride, input_hw):
        super().__init__(
            StaticSamePadConv2d(
                channels,
                channels,
                kernel_size=kernel_size,
                stride=stride,
                groups=channels,
                bias=False,
                input_hw=input_hw,
            ),
            nn.BatchNorm2d(channels, eps=1e-3),
            nn.ReLU6(inplace=True),
        )


class ProjectConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels, eps=1e-3),
        )


class InvertedResidual(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride,
        expand_ratio,
        input_hw,
        use_residual,
    ):
        super().__init__()

        hidden_channels = int(round(in_channels * expand_ratio))
        self.use_residual = use_residual

        layers = []

        if expand_ratio != 1:
            layers.append(
                ConvBNReLU6(
                    in_channels,
                    hidden_channels,
                    kernel_size=1,
                    stride=1,
                    input_hw=input_hw,
                )
            )

        layers.append(
            DepthwiseConvBNReLU6(
                hidden_channels,
                kernel_size=3,
                stride=stride,
                input_hw=input_hw,
            )
        )

        layers.append(ProjectConvBN(hidden_channels, out_channels))

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv(x)

        if self.use_residual:
            out = x + out

        return out


class LibreFOMOBackbone(nn.Module):
    def __init__(self, size: str):
        super().__init__()

        if size not in CONFIGS:
            raise ValueError(f"Unsupported LibreFOMO size: {size!r}")

        cfg = CONFIGS[size]
        alpha = cfg["alpha"]
        imgsz = cfg["imgsz"]

        c0 = _make_divisible(32 * alpha, 8)
        c1 = _make_divisible(16 * alpha, 8)
        c2 = _make_divisible(24 * alpha, 8)
        c3 = _make_divisible(32 * alpha, 8)

        self.conv1 = ConvBNReLU6(
            3,
            c0,
            kernel_size=3,
            stride=2,
            input_hw=(imgsz, imgsz),
        )

        hw = math.ceil(imgsz / 2)

        self.expanded_conv = InvertedResidual(
            c0,
            c1,
            stride=1,
            expand_ratio=1,
            input_hw=(hw, hw),
            use_residual=False,
        )

        self.block_1 = InvertedResidual(
            c1,
            c2,
            stride=2,
            expand_ratio=6,
            input_hw=(hw, hw),
            use_residual=False,
        )

        hw = math.ceil(hw / 2)

        self.block_2 = InvertedResidual(
            c2,
            c2,
            stride=1,
            expand_ratio=6,
            input_hw=(hw, hw),
            use_residual=True,
        )

        self.block_3 = InvertedResidual(
            c2,
            c3,
            stride=2,
            expand_ratio=6,
            input_hw=(hw, hw),
            use_residual=False,
        )

        hw = math.ceil(hw / 2)

        self.block_4 = InvertedResidual(
            c3,
            c3,
            stride=1,
            expand_ratio=6,
            input_hw=(hw, hw),
            use_residual=True,
        )

        self.block_5 = InvertedResidual(
            c3,
            c3,
            stride=1,
            expand_ratio=6,
            input_hw=(hw, hw),
            use_residual=True,
        )

        self.block_6_expand = ConvBNReLU6(
            c3,
            int(round(c3 * 6)),
            kernel_size=1,
            stride=1,
            input_hw=(hw, hw),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.expanded_conv(x)
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = self.block_4(x)
        x = self.block_5(x)
        x = self.block_6_expand(x)
        return x


class LibreFOMO(nn.Module):
    def __init__(self, size: str = "m", nc: int = 1, head_channels: int = 2):
        super().__init__()

        if size not in CONFIGS:
            raise ValueError(f"Unsupported LibreFOMO size: {size!r}")

        self.size = size
        self.nc = nc
        self.head_channels = head_channels
        self.imgsz = CONFIGS[size]["imgsz"]

        self.backbone = LibreFOMOBackbone(size)

        self.head = nn.Conv2d(
            CONFIGS[size]["head_in_channels"],
            head_channels,
            kernel_size=1,
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def load_librefomo_checkpoint(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)

    required = {
        "format_version": 2,
        "arch": "LibreFOMO",
        "model_family": "librefomo",
        "backbone": "mobilenet_v2_keras_tf_same_staticpad_direct_block6_expand",
        "state_dict_key": "model",
    }

    for key, expected in required.items():
        actual = ckpt.get(key)
        if actual != expected:
            raise ValueError(
                f"Invalid checkpoint metadata {key}: expected {expected!r}, got {actual!r}"
            )

    if "model" not in ckpt:
        raise KeyError("Checkpoint is missing state dict key 'model'")

    model = LibreFOMO(
        size=ckpt["size"],
        nc=ckpt["nc"],
        head_channels=ckpt["head_channels"],
    )

    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    return model, ckpt
'''.strip() + "\n"

MODEL_PATH = OUTDIR / "model.py"
MODEL_PATH.write_text(MODEL_PY)

spec = importlib.util.spec_from_file_location("librefomo_model", MODEL_PATH)
runtime = importlib.util.module_from_spec(spec)
sys.modules["librefomo_model"] = runtime
spec.loader.exec_module(runtime)


# ----------------------------
# Conversion helpers
# ----------------------------

def assign_conv2d(pt_conv: nn.Conv2d, keras_w: np.ndarray):
    pt_conv.weight.data.copy_(torch.from_numpy(keras_w.transpose(3, 2, 0, 1)))


def assign_depthwise(pt_conv: nn.Conv2d, keras_w: np.ndarray):
    pt_conv.weight.data.copy_(torch.from_numpy(keras_w.transpose(2, 3, 0, 1)))


def assign_bn(pt_bn: nn.BatchNorm2d, keras_bn_weights: List[np.ndarray]):
    gamma, beta, mean, var = keras_bn_weights

    pt_bn.weight.data.copy_(torch.from_numpy(gamma))
    pt_bn.bias.data.copy_(torch.from_numpy(beta))
    pt_bn.running_mean.data.copy_(torch.from_numpy(mean))
    pt_bn.running_var.data.copy_(torch.from_numpy(var))


def conv_of(block):
    return block[0].conv


def bn_of(block):
    return block[1]


def download_if_needed(url: str, path: Path):
    if not path.exists():
        print(f"Downloading {path.name}")
        urllib.request.urlretrieve(url, path)


def build_keras_model(size: str, h5_path: Path):
    cfg = CONFIGS[size]

    return tf.keras.applications.MobileNetV2(
        input_shape=(cfg["imgsz"], cfg["imgsz"], 3),
        alpha=cfg["alpha"],
        include_top=False,
        weights=str(h5_path),
    )


def copy_keras_to_clean_backbone(keras_model: tf.keras.Model, pt_backbone: nn.Module):
    def kw(layer_name: str):
        return keras_model.get_layer(layer_name).get_weights()

    # Conv1
    assign_conv2d(conv_of(pt_backbone.conv1), kw("Conv1")[0])
    assign_bn(bn_of(pt_backbone.conv1), kw("bn_Conv1"))

    # expanded_conv: depthwise -> project
    assign_depthwise(
        conv_of(pt_backbone.expanded_conv.conv[0]),
        kw("expanded_conv_depthwise")[0],
    )
    assign_bn(
        bn_of(pt_backbone.expanded_conv.conv[0]),
        kw("expanded_conv_depthwise_BN"),
    )
    assign_conv2d(
        pt_backbone.expanded_conv.conv[1][0],
        kw("expanded_conv_project")[0],
    )
    assign_bn(
        pt_backbone.expanded_conv.conv[1][1],
        kw("expanded_conv_project_BN"),
    )

    # block_1 through block_5: expand -> depthwise -> project
    for idx in range(1, 6):
        block = getattr(pt_backbone, f"block_{idx}")

        assign_conv2d(
            conv_of(block.conv[0]),
            kw(f"block_{idx}_expand")[0],
        )
        assign_bn(
            bn_of(block.conv[0]),
            kw(f"block_{idx}_expand_BN"),
        )

        assign_depthwise(
            conv_of(block.conv[1]),
            kw(f"block_{idx}_depthwise")[0],
        )
        assign_bn(
            bn_of(block.conv[1]),
            kw(f"block_{idx}_depthwise_BN"),
        )

        assign_conv2d(
            block.conv[2][0],
            kw(f"block_{idx}_project")[0],
        )
        assign_bn(
            block.conv[2][1],
            kw(f"block_{idx}_project_BN"),
        )

    # block_6_expand only
    assign_conv2d(
        conv_of(pt_backbone.block_6_expand),
        kw("block_6_expand")[0],
    )
    assign_bn(
        bn_of(pt_backbone.block_6_expand),
        kw("block_6_expand_BN"),
    )


def initialize_head(model: nn.Module, seed: int = SEED):
    torch.manual_seed(seed)

    nn.init.kaiming_uniform_(model.head.weight, a=math.sqrt(5))

    if model.head.bias is not None:
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(model.head.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(model.head.bias, -bound, bound)


def save_checkpoint(path: Path, model: nn.Module, size: str):
    cfg = CONFIGS[size]

    state_dict = OrderedDict(
        (key, value.detach().cpu().clone())
        for key, value in model.state_dict().items()
    )

    from libreyolo.utils.serialization import wrap_libreyolo_checkpoint

    ckpt = wrap_libreyolo_checkpoint(
        state_dict,
        model_family="librefomo",
        size=size,
        task="point",
        nc=model.nc,
        names={0: "person"},
        imgsz=cfg["imgsz"],
        format_version=2,
        arch="LibreFOMO",
        backbone="mobilenet_v2_keras_tf_same_staticpad_direct_block6_expand",
        alpha=cfg["alpha"],
        head_channels=model.head_channels,
        supported_tasks=["point", "detect"],
        default_task="point",
        state_dict_key="model",
    )

    torch.save(ckpt, path)


def verify_backbone_similarity(size: str, keras_model: tf.keras.Model, pt_model: nn.Module):
    cfg = CONFIGS[size]
    imgsz = cfg["imgsz"]

    keras_probe = tf.keras.Model(
        inputs=keras_model.input,
        outputs=keras_model.get_layer("block_6_expand_relu").output,
    )

    rng = np.random.default_rng(SEED + len(size) + imgsz)
    x_nhwc = rng.normal(size=(1, imgsz, imgsz, 3)).astype(np.float32)
    x_nchw = np.transpose(x_nhwc, (0, 3, 1, 2))

    keras_out = keras_probe(x_nhwc, training=False).numpy()
    torch_out = pt_model.backbone(torch.from_numpy(x_nchw)).detach().cpu().numpy()
    torch_out = np.transpose(torch_out, (0, 2, 3, 1))

    max_abs = float(np.max(np.abs(keras_out - torch_out)))
    mean_abs = float(np.mean(np.abs(keras_out - torch_out)))

    if max_abs > VERIFY_ATOL_MAX:
        raise AssertionError(
            f"Backbone similarity check failed for size={size}: "
            f"max_abs={max_abs:.8f}, mean_abs={mean_abs:.8f}, "
            f"tolerance={VERIFY_ATOL_MAX}"
        )

    print(
        f"Verified Keras similarity size={size}: "
        f"max_abs={max_abs:.8f}, mean_abs={mean_abs:.8f}"
    )


def convert_one_size(size: str):
    cfg = CONFIGS[size]

    weights_dir = OUTDIR / "keras_weights"
    weights_dir.mkdir(parents=True, exist_ok=True)

    h5_path = weights_dir / f"mobilenet_v2_{size}_no_top.h5"
    download_if_needed(cfg["url"], h5_path)

    keras_model = build_keras_model(size, h5_path)

    model = runtime.LibreFOMO(size=size, nc=1, head_channels=2)
    model.eval()

    copy_keras_to_clean_backbone(keras_model, model.backbone)
    initialize_head(model, seed=SEED)
    model.eval()

    verify_backbone_similarity(size, keras_model, model)

    ckpt_path = OUTDIR / cfg["out"]
    save_checkpoint(ckpt_path, model, size)

    return ckpt_path


# ----------------------------
# Generate checkpoints
# ----------------------------

ckpt_paths = [convert_one_size(size) for size in SIZES]


# ----------------------------
# Strict validation using generated model.py
# ----------------------------

for ckpt_path in ckpt_paths:
    model, ckpt = runtime.load_librefomo_checkpoint(ckpt_path, map_location="cpu")

    with torch.no_grad():
        out = model(torch.zeros(1, 3, ckpt["imgsz"], ckpt["imgsz"]))

    print(f"Strict load OK: {ckpt_path} -> output shape {tuple(out.shape)}")


Verified Keras similarity size=s: max_abs=0.00001097, mean_abs=0.00000070
Verified Keras similarity size=m: max_abs=0.00001597, mean_abs=0.00000055
Verified Keras similarity size=l: max_abs=0.00001366, mean_abs=0.00000035
Strict load OK: /content/librefomo_export/LibreFOMOs.pt -> output shape (1, 2, 12, 12)
Strict load OK: /content/librefomo_export/LibreFOMOm.pt -> output shape (1, 2, 24, 24)
Strict load OK: /content/librefomo_export/LibreFOMOl.pt -> output shape (1, 2, 32, 32)


In [14]:
# ----------------------------
# One sample load/inference
# ----------------------------

sample_path = OUTDIR / "LibreFOMOm.pt"
sample_model, sample_ckpt = runtime.load_librefomo_checkpoint(sample_path, map_location="cpu")

with torch.no_grad():
    sample_out = sample_model(
        torch.zeros(1, 3, sample_ckpt["imgsz"], sample_ckpt["imgsz"])
    )

print(f"Sample inference OK: {sample_path} -> output shape {tuple(sample_out.shape)}")
print(f"Wrote: {OUTDIR / 'LibreFOMOs.pt'}")
print(f"Wrote: {OUTDIR / 'LibreFOMOm.pt'}")
print(f"Wrote: {OUTDIR / 'LibreFOMOl.pt'}")
print(f"Wrote: {OUTDIR / 'model.py'}")

Sample inference OK: /content/librefomo_export/LibreFOMOm.pt -> output shape (1, 2, 24, 24)
Wrote: /content/librefomo_export/LibreFOMOs.pt
Wrote: /content/librefomo_export/LibreFOMOm.pt
Wrote: /content/librefomo_export/LibreFOMOl.pt
Wrote: /content/librefomo_export/model.py


In [15]:
# pip install torch huggingface_hub

from huggingface_hub import hf_hub_download
import importlib.util
import torch

repo_id = "bdanko/LibreFOMO"

# Download model.py and checkpoint from HF
model_py = hf_hub_download(repo_id=repo_id, filename="model.py")
ckpt_path = hf_hub_download(repo_id=repo_id, filename="LibreFOMOm.pt")

# Import the downloaded model.py
spec = importlib.util.spec_from_file_location("librefomo_model", model_py)
librefomo_model = importlib.util.module_from_spec(spec)
spec.loader.exec_module(librefomo_model)

# Load checkpoint
model, ckpt = librefomo_model.load_librefomo_checkpoint(ckpt_path, map_location="cpu")

# Smoke-test inference
x = torch.zeros(1, 3, ckpt["imgsz"], ckpt["imgsz"])
with torch.no_grad():
    y = model(x)

print(y.shape)
print(ckpt["size"], ckpt["imgsz"], ckpt["names"])

model.py:   0%|          | 0.00/8.94k [00:00<?, ?B/s]

LibreFOMOm.pt:   0%|          | 0.00/133k [00:00<?, ?B/s]

torch.Size([1, 2, 24, 24])
m 192 {0: 'person'}
